# 01 · Fetch & chip Sentinel-2 imagery — Front Range, CO

Searches Microsoft Planetary Computer for a low-cloud Sentinel-2 L2A mosaic over the Boulder/Denver Front Range, cuts it into a grid of 224x224px (2.24km) chips, and saves the raw pixel arrays + chip geolocation metadata for notebook 02 to embed.

**Runtime:** CPU is fine for this notebook, but if you're running 01->03 in one sitting, switch to a GPU runtime now (Runtime -> Change runtime type -> T4 GPU) so you don't have to restart for notebook 02.

**Before running:** push this repo to GitHub, then set `REPO_URL` below to your repo's clone URL. Colab's "Open from GitHub" only loads this single `.ipynb` file, not the rest of the repo (`src/`, `docs/`) — the clone cell below fetches everything else.

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import json
import sys

sys.path.append(os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import odc.stac

from src import stac_utils

# Each notebook you open from the GitHub link gets its own fresh Colab VM --
# notebook 02 opened separately would NOT see files saved to this VM's local
# disk. Mounting Drive gives all three notebooks a shared, persistent place
# to hand data off to each other regardless of which VM each one lands on.
from google.colab import drive

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/SATEMB_data"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)

## Search Sentinel-2 L2A over the AOI

The AOI (`stac_utils.FRONT_RANGE_BBOX`) spans Boulder -> Denver, foothills -> plains, and includes Boulder Reservoir plus the Dec-2021 Marshall Fire burn scar near Superior/Louisville — deliberately picked for embedding diversity (urban grid, suburban, farmland, forest, water, a fire scar).

Rather than building a full seasonal median composite (which for an AOI this size means downloading tens of GB across dozens of revisits), we take the handful of least-cloudy scenes that together cover the bbox and median them — a light composite that's robust to a stray cloud or shadow without an unreasonable Colab download. A true multi-month composite is a nice extension exercise if you want to push this further.

In [ ]:
catalog = stac_utils.open_catalog()
items = stac_utils.search_sentinel2(catalog)
items_sorted = sorted(items, key=lambda it: it.properties.get("eo:cloud_cover", 100))

print(f"{len(items_sorted)} candidate scenes found")
for it in items_sorted[:5]:
    print(f"  {it.id}  cloud_cover={it.properties.get('eo:cloud_cover'):.1f}%  date={it.datetime.date()}")

# Take the 3 least-cloudy scenes -- in practice this AOI is covered by 1-2
# Sentinel-2 MGRS tiles, so 3 dates gives enough overlap to median away
# residual cloud/shadow at tile seams without ballooning download size.
mosaic_items = items_sorted[:3]

## Load and composite

Reprojected to UTM 13N (`EPSG:32613`, meters) rather than left in lon/lat degrees, so a 224px chip is an exact, undistorted 2.24km square everywhere in the AOI.

In [ ]:
ds = odc.stac.load(
    mosaic_items,
    bands=stac_utils.S2_BANDS,
    bbox=stac_utils.FRONT_RANGE_BBOX,
    crs="EPSG:32613",
    resolution=stac_utils.GSD_M,
    groupby="solar_day",
    chunks={"x": 1024, "y": 1024},
)

# (time, y, x) per band -> (band, y, x) median composite
mosaic = ds.to_array(dim="band").median(dim="time").compute()
print(mosaic.shape, mosaic.dtype)

In [ ]:
# Quick sanity check: a true-color composite of the whole AOI. You should
# recognize Denver's street grid in the SE, forested foothills along the
# west edge, and farmland/plains texture to the east.
rgb = mosaic.sel(band=["B04", "B03", "B02"]).values.astype("float32")
rgb = np.clip(rgb / np.percentile(rgb, 98), 0, 1).transpose(1, 2, 0)

plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.title(f"Front Range mosaic — {mosaic.sizes['y']}x{mosaic.sizes['x']} px")
plt.axis("off")
plt.savefig("outputs/figures/aoi_true_color.png", dpi=150, bbox_inches="tight")
plt.show()

## Cut into chips

Each chip becomes one polygon on the final map and one row in the embedding matrix. We drop chips that are mostly nodata (can happen at bbox/tile edges).

In [ ]:
height, width = mosaic.sizes["y"], mosaic.sizes["x"]
grid = stac_utils.make_pixel_chip_grid(height, width)
print(f"{len(grid)} candidate chips ({height // stac_utils.CHIP_SIZE_PX} rows x {width // stac_utils.CHIP_SIZE_PX} cols)")

x_coords = mosaic.x.values
y_coords = mosaic.y.values
raster_crs = ds.odc.crs
acquisition_date = str(mosaic_items[0].datetime.date())

chip_arrays = []
chips_meta = []
NODATA_FRAC_THRESHOLD = 0.05

for chip in grid:
    arr = mosaic.isel(y=chip["y_slice"], x=chip["x_slice"]).values
    nodata_frac = np.isnan(arr).mean()
    if nodata_frac > NODATA_FRAC_THRESHOLD:
        continue
    arr = np.nan_to_num(arr, nan=0.0)

    bounds = stac_utils.pixel_window_to_lonlat_bounds(x_coords, y_coords, chip, raster_crs)
    lat, lon = stac_utils.bounds_centroid(bounds)

    chip_arrays.append(arr.astype("float32"))
    chips_meta.append(
        {
            "id": chip["id"],
            "bounds": bounds,
            "lat": lat,
            "lon": lon,
            "date": acquisition_date,
        }
    )

chip_pixels = np.stack(chip_arrays)  # (N, 10, 224, 224)
print(f"Kept {len(chips_meta)}/{len(grid)} chips -> pixel array {chip_pixels.shape}, {chip_pixels.nbytes / 1e9:.2f} GB")

In [ ]:
np.save(f"{DATA_DIR}/chip_pixels.npy", chip_pixels)
with open(f"{DATA_DIR}/chips_meta.json", "w") as f:
    json.dump(chips_meta, f)

print(f"Saved chip_pixels.npy and chips_meta.json to {DATA_DIR}")
print("Next: run notebooks/02_generate_embeddings.ipynb")